---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # m_DIM_CUSTOMER_SCD2 (converted from Informatica PowerCenter)
    SCD Type 2 load. Informatica writes row by row with Update Strategy (DD_INSERT / DD_UPDATE). The Designer pipeline computes the **final state** of `DIM_CUSTOMER` (expired old versions + unchanged rows + new versions) and overwrites the target. Alternative: Output in `merge` mode keyed on `customer_sk`.

    | Informatica | Lakeflow Designer |
    |---|---|
    | `$$LOAD_DATE` (parameter file) | `env_config` |
    | SQ_CUSTOMER_UPDATES (filter EXTRACT_DATE = $$LOAD_DATE) | `sq_updates_filter` (SQL, needs the parameter) |
    | mplt_STD_CUSTOMER (mapplet, inlined) | `mplt_std_customer` |
    | EXP_HASH (MD5) | `exp_hash` |
    | LKP_DIM_CURRENT (SQL override, CURRENT_FLAG = 'Y') | `lkp_dim_filter` + `lkp_dim_hash` + `lkp_dim_current` |
    | EXP_CHANGE_FLAG | `exp_change_flag` |
    | RTR_CHANGE NEW / CHANGED (+ SEQ_CUSTOMER_SK, UPD_INSERT_*) | `rtr_new_or_changed` + `seq_new_versions` |
    | RTR_CHANGE CHANGED -> UPD_EXPIRE_OLD (DD_UPDATE) | `rtr_changed` + `upd_join_expired` + `upd_expire_old` |
    | 3 target instances of DIM_CUSTOMER | `align_new_versions` + `union_dim` + `out_dim_customer` |

    **Sequence**: Current Value = 1001, so new keys start at 1001. Informatica hands out keys in row-arrival order; the pipeline orders by `customer_id` (deterministic). Confirm with the owner if the key order matters.

    **MANUAL (Lakeflow Jobs)**: session Pre SQL (`ALTER SESSION SET NLS_DATE_FORMAT`, not needed), Post SQL (`DBMS_STATS`, use `ANALYZE TABLE` / predictive optimization), On Failure Email (job notification), parameter file (job parameter).
---

In [0]:
"""
id: "env_config"
template: "python"
templateVersion: "1.0.0"
name: "env_config"
position:
  x: 0
  y: 360
description:
  text: "Mapping parameter $$LOAD_DATE."
  hash: ""
previewMode: "1000"
config:
  code: |-
    # $$LOAD_DATE from $PMRootDir/params/dim_customer.par
    LOAD_DATE = "2026-10-31"
    result = spark.createDataFrame([(LOAD_DATE,)], ["load_date"])
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    # $$LOAD_DATE from $PMRootDir/params/dim_customer.par
    LOAD_DATE = "2026-10-31"
    result = spark.createDataFrame([(LOAD_DATE,)], ["load_date"])

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {
    "data": [],
    "data__sources": []
}
out = run(config, inputs, spark)
ctx["env_config.result"] = out["result"]


In [0]:
"""
id: "src_customer_updates"
template: "source"
templateVersion: "1.0.0"
name: "src_customer_updates"
position:
  x: 0
  y: 515
description:
  text: "SQ_CUSTOMER_UPDATES source (STAGING.CUSTOMER_UPDATES)."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_customer_updates"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_customer_updates"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_customer_updates.data"] = out["data"]


In [0]:
"""
id: "src_dim_customer"
template: "source"
templateVersion: "1.0.0"
name: "src_dim_customer"
position:
  x: 0
  y: 670
description:
  text: "DIM_CUSTOMER (current target state; lookup + update target)."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_dim_customer"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_dim_customer"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_dim_customer.data"] = out["data"]


In [0]:
"""
id: "sq_updates_filter"
template: "sql"
templateVersion: "1.0.0"
name: "sq_updates_filter"
position:
  x: 300
  y: 360
description:
  text: "SQ Source Filter: EXTRACT_DATE = TO_DATE('$$LOAD_DATE','YYYY-MM-DD')."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT u.*
    FROM src_customer_updates u
    CROSS JOIN env_config c
    WHERE u.extract_date = CAST(c.load_date AS DATE)
input:
  - node: "src_customer_updates"
    input_port: "data"
    output_port: "data"
  - node: "env_config"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT u.*\nFROM src_customer_updates u\nCROSS JOIN env_config c\nWHERE u.extract_date = CAST(c.load_date AS DATE)"
}
inputs = {
    "data": [
        ctx["src_customer_updates.data"],
        ctx["env_config.result"]
    ],
    "data__sources": [
        {
            "node": "src_customer_updates",
            "output_port": "data",
            "name": "src_customer_updates",
            "df_name": "src_customer_updates"
        },
        {
            "node": "env_config",
            "output_port": "result",
            "name": "env_config",
            "df_name": "env_config"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sq_updates_filter.result"] = out["result"]


In [0]:
"""
id: "mplt_std_customer"
template: "transform"
templateVersion: "1.0.0"
name: "mplt_std_customer"
position:
  x: 600
  y: 360
description:
  text: "mplt_STD_CUSTOMER (mapplet EXP_STD inlined)."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "LOWER(TRIM(email)) AS email"
    - "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS segment"
    - "UPPER(TRIM(country)) AS country"
input:
  - node: "sq_updates_filter"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "LOWER(TRIM(email)) AS email",
        "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS segment",
        "UPPER(TRIM(country)) AS country"
    ]
}
inputs = {
    "data": ctx["sq_updates_filter.result"]
}
out = run(config, inputs, spark)
ctx["mplt_std_customer.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "exp_hash"
template: "transform"
templateVersion: "1.0.0"
name: "exp_hash"
position:
  x: 900
  y: 360
description:
  text: "EXP_HASH: MD5 over the tracked attributes (|| treats NULL as '' like Informatica)."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "MD5(COALESCE(email, '') || '|' || COALESCE(segment, '') || '|' || COALESCE(country, '')) AS src_hash"
input:
  - node: "mplt_std_customer"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "MD5(COALESCE(email, '') || '|' || COALESCE(segment, '') || '|' || COALESCE(country, '')) AS src_hash"
    ]
}
inputs = {
    "data": ctx["mplt_std_customer.transformed_data"]
}
out = run(config, inputs, spark)
ctx["exp_hash.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "lkp_dim_filter"
template: "filter"
templateVersion: "1.0.0"
name: "lkp_dim_filter"
position:
  x: 300
  y: 515
description:
  text: "LKP_DIM_CURRENT SQL override: WHERE CURRENT_FLAG = 'Y'."
  hash: ""
previewMode: "1000"
config:
  condition: "current_flag = 'Y'"
input:
  - node: "src_dim_customer"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "current_flag = 'Y'"
}
inputs = {
    "data": ctx["src_dim_customer.data"]
}
out = run(config, inputs, spark)
ctx["lkp_dim_filter.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "lkp_dim_hash"
template: "transform"
templateVersion: "1.0.0"
name: "lkp_dim_hash"
position:
  x: 600
  y: 515
description:
  text: "LKP_DIM_CURRENT SQL override: DIM_HASH = MD5(EMAIL||'|'||SEGMENT||'|'||COUNTRY)."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_sk"
    - "customer_id"
    - "MD5(COALESCE(email, '') || '|' || COALESCE(segment, '') || '|' || COALESCE(country, '')) AS dim_hash"
input:
  - node: "lkp_dim_filter"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_sk",
        "customer_id",
        "MD5(COALESCE(email, '') || '|' || COALESCE(segment, '') || '|' || COALESCE(country, '')) AS dim_hash"
    ]
}
inputs = {
    "data": ctx["lkp_dim_filter.filtered_data"]
}
out = run(config, inputs, spark)
ctx["lkp_dim_hash.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "lkp_dim_current"
template: "join"
templateVersion: "1.0.0"
name: "lkp_dim_current"
position:
  x: 1200
  y: 360
description:
  text: "LKP_DIM_CURRENT: connected lookup on CUSTOMER_ID (policy Report Error: at most one current row per customer)."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`customer_id` = right.`customer_id`"
  expressions:
    - "left.*"
    - "right.`customer_sk` AS `customer_sk`"
    - "right.`dim_hash` AS `dim_hash`"
input:
  - node: "exp_hash"
    input_port: "left"
    output_port: "transformed_data"
  - node: "lkp_dim_hash"
    input_port: "right"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`customer_id` = right.`customer_id`",
    "expressions": [
        "left.*",
        "right.`customer_sk` AS `customer_sk`",
        "right.`dim_hash` AS `dim_hash`"
    ]
}
inputs = {
    "left": ctx["exp_hash.transformed_data"],
    "right": ctx["lkp_dim_hash.transformed_data"]
}
out = run(config, inputs, spark)
ctx["lkp_dim_current.joined_data"] = out["joined_data"]


In [0]:
"""
id: "exp_change_flag"
template: "transform"
templateVersion: "1.0.0"
name: "exp_change_flag"
position:
  x: 1500
  y: 360
description:
  text: "EXP_CHANGE_FLAG: IIF(ISNULL(CUSTOMER_SK),'NEW',IIF(o_SRC_HASH != DIM_HASH,'CHANGED','NOCHANGE'))."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "CASE WHEN customer_sk IS NULL THEN 'NEW' WHEN src_hash <> dim_hash THEN 'CHANGED' ELSE 'NOCHANGE' END AS change_flag"
input:
  - node: "lkp_dim_current"
    input_port: "data"
    output_port: "joined_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "CASE WHEN customer_sk IS NULL THEN 'NEW' WHEN src_hash <> dim_hash THEN 'CHANGED' ELSE 'NOCHANGE' END AS change_flag"
    ]
}
inputs = {
    "data": ctx["lkp_dim_current.joined_data"]
}
out = run(config, inputs, spark)
ctx["exp_change_flag.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "rtr_new_or_changed"
template: "filter"
templateVersion: "1.0.0"
name: "rtr_new_or_changed"
position:
  x: 1800
  y: 360
description:
  text: "RTR_CHANGE groups NEW + CHANGED -> insert a new version."
  hash: ""
previewMode: "1000"
config:
  condition: "change_flag IN ('NEW', 'CHANGED')"
input:
  - node: "exp_change_flag"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "change_flag IN ('NEW', 'CHANGED')"
}
inputs = {
    "data": ctx["exp_change_flag.transformed_data"]
}
out = run(config, inputs, spark)
ctx["rtr_new_or_changed.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "rtr_changed"
template: "filter"
templateVersion: "1.0.0"
name: "rtr_changed"
position:
  x: 1800
  y: 515
description:
  text: "RTR_CHANGE group CHANGED -> expire the old version."
  hash: ""
previewMode: "1000"
config:
  condition: "change_flag = 'CHANGED'"
input:
  - node: "exp_change_flag"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "change_flag = 'CHANGED'"
}
inputs = {
    "data": ctx["exp_change_flag.transformed_data"]
}
out = run(config, inputs, spark)
ctx["rtr_changed.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "seq_new_versions"
template: "sql"
templateVersion: "1.0.0"
name: "seq_new_versions"
position:
  x: 2100
  y: 360
description:
  text: "SEQ_CUSTOMER_SK (shared, Current Value 1001) + EXP_NEW_VERSION(_CHG) + UPD_INSERT_* (DD_INSERT)."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT 1000 + ROW_NUMBER() OVER (ORDER BY n.customer_id) AS customer_sk,
           n.customer_id, n.email, n.segment, n.country,
           CAST(c.load_date AS DATE) AS valid_from,
           DATE'9999-12-31' AS valid_to,
           'Y' AS current_flag
    FROM rtr_new_or_changed n
    CROSS JOIN env_config c
input:
  - node: "rtr_new_or_changed"
    input_port: "data"
    output_port: "filtered_data"
  - node: "env_config"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT 1000 + ROW_NUMBER() OVER (ORDER BY n.customer_id) AS customer_sk,\n       n.customer_id, n.email, n.segment, n.country,\n       CAST(c.load_date AS DATE) AS valid_from,\n       DATE'9999-12-31' AS valid_to,\n       'Y' AS current_flag\nFROM rtr_new_or_changed n\nCROSS JOIN env_config c"
}
inputs = {
    "data": [
        ctx["rtr_new_or_changed.filtered_data"],
        ctx["env_config.result"]
    ],
    "data__sources": [
        {
            "node": "rtr_new_or_changed",
            "output_port": "filtered_data",
            "name": "rtr_new_or_changed",
            "df_name": "rtr_new_or_changed"
        },
        {
            "node": "env_config",
            "output_port": "result",
            "name": "env_config",
            "df_name": "env_config"
        }
    ]
}
out = run(config, inputs, spark)
ctx["seq_new_versions.result"] = out["result"]


In [0]:
"""
id: "upd_join_expired"
template: "join"
templateVersion: "1.0.0"
name: "upd_join_expired"
position:
  x: 2100
  y: 515
description:
  text: "Mark the dimension rows targeted by UPD_EXPIRE_OLD (DD_UPDATE keyed on CUSTOMER_SK)."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`customer_sk` = right.`customer_sk`"
  expressions:
    - "left.*"
    - "right.`customer_sk` AS `expired_sk`"
input:
  - node: "src_dim_customer"
    input_port: "left"
    output_port: "data"
  - node: "rtr_changed"
    input_port: "right"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`customer_sk` = right.`customer_sk`",
    "expressions": [
        "left.*",
        "right.`customer_sk` AS `expired_sk`"
    ]
}
inputs = {
    "left": ctx["src_dim_customer.data"],
    "right": ctx["rtr_changed.filtered_data"]
}
out = run(config, inputs, spark)
ctx["upd_join_expired.joined_data"] = out["joined_data"]


In [0]:
"""
id: "upd_expire_old"
template: "sql"
templateVersion: "1.0.0"
name: "upd_expire_old"
position:
  x: 2400
  y: 360
description:
  text: "UPD_EXPIRE_OLD: VALID_TO = ADD_TO_DATE($$LOAD_DATE,'DD',-1), CURRENT_FLAG = 'N'."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT d.customer_sk, d.customer_id, d.email, d.segment, d.country, d.valid_from,
           CASE WHEN d.expired_sk IS NOT NULL THEN DATE_SUB(CAST(c.load_date AS DATE), 1) ELSE d.valid_to END AS valid_to,
           CASE WHEN d.expired_sk IS NOT NULL THEN 'N' ELSE d.current_flag END AS current_flag
    FROM upd_join_expired d
    CROSS JOIN env_config c
input:
  - node: "upd_join_expired"
    input_port: "data"
    output_port: "joined_data"
  - node: "env_config"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT d.customer_sk, d.customer_id, d.email, d.segment, d.country, d.valid_from,\n       CASE WHEN d.expired_sk IS NOT NULL THEN DATE_SUB(CAST(c.load_date AS DATE), 1) ELSE d.valid_to END AS valid_to,\n       CASE WHEN d.expired_sk IS NOT NULL THEN 'N' ELSE d.current_flag END AS current_flag\nFROM upd_join_expired d\nCROSS JOIN env_config c"
}
inputs = {
    "data": [
        ctx["upd_join_expired.joined_data"],
        ctx["env_config.result"]
    ],
    "data__sources": [
        {
            "node": "upd_join_expired",
            "output_port": "joined_data",
            "name": "upd_join_expired",
            "df_name": "upd_join_expired"
        },
        {
            "node": "env_config",
            "output_port": "result",
            "name": "env_config",
            "df_name": "env_config"
        }
    ]
}
out = run(config, inputs, spark)
ctx["upd_expire_old.result"] = out["result"]


In [0]:
"""
id: "align_new_versions"
template: "transform"
templateVersion: "1.0.0"
name: "align_new_versions"
position:
  x: 2400
  y: 515
description:
  text: "Column order and types aligned with the dimension (Combine unions by position)."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "CAST(customer_sk AS INT) AS customer_sk"
    - "customer_id"
    - "email"
    - "segment"
    - "country"
    - "valid_from"
    - "valid_to"
    - "current_flag"
input:
  - node: "seq_new_versions"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "CAST(customer_sk AS INT) AS customer_sk",
        "customer_id",
        "email",
        "segment",
        "country",
        "valid_from",
        "valid_to",
        "current_flag"
    ]
}
inputs = {
    "data": ctx["seq_new_versions.result"]
}
out = run(config, inputs, spark)
ctx["align_new_versions.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "union_dim"
template: "combine"
name: "union_dim"
position:
  x: 2700
  y: 360
description:
  text: "Final DIM_CUSTOMER state = existing rows (with expirations) + new versions."
  hash: ""
previewMode: "1000"
config:
  operator: "UNION"
  quantifier: "ALL"
input:
  - node: "upd_expire_old"
    input_port: "data_0"
    output_port: "result"
  - node: "align_new_versions"
    input_port: "data_1"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df_0 = inputs["data_0"]
    df_1 = inputs["data_1"]
    operator = config.get("operator", "UNION")
    quantifier = config.get("quantifier", "DISTINCT")

    op_map = {
        "UNION": lambda a, b: a.union(b),
        "INTERSECT": lambda a, b: a.intersectAll(b),
        "EXCEPT": lambda a, b: a.exceptAll(b),
        "MINUS": lambda a, b: a.exceptAll(b),
    }
    op_distinct_map = {
        "UNION": lambda a, b: a.union(b).distinct(),
        "INTERSECT": lambda a, b: a.intersect(b),
        "EXCEPT": lambda a, b: a.subtract(b),
        "MINUS": lambda a, b: a.subtract(b),
    }
    if quantifier == "DISTINCT":
        combine_fn = op_distinct_map.get(operator)
    else:
        combine_fn = op_map.get(operator)
    if not combine_fn:
        raise ValueError(f"Unsupported combine operator: {operator}")

    return {"combined_data": combine_fn(df_0, df_1)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "operator": "UNION",
    "quantifier": "ALL"
}
inputs = {
    "data_0": ctx["upd_expire_old.result"],
    "data_1": ctx["align_new_versions.transformed_data"]
}
out = run(config, inputs, spark)
ctx["union_dim.combined_data"] = out["combined_data"]


In [0]:
"""
id: "out_dim_customer"
template: "output"
templateVersion: "1.0.0"
name: "out_dim_customer"
position:
  x: 3000
  y: 360
description:
  text: "DIM_CUSTOMER (final SCD2 state)."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "infa_05_dim_customer"
input:
  - node: "union_dim"
    input_port: "data"
    output_port: "combined_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "infa_05_dim_customer"
}
inputs = {
    "data": ctx["union_dim.combined_data"]
}
out = run(config, inputs, spark)
